# 40.04 КТ/FEM-чувствительность ТТРКГ — реокардиомонитор МГТУ

Ноутбук принимает внешний FEM-оператор только при точном совпадении записи,
дыхательного состояния, монтажа, электродной геометрии, частоты, материалов,
версии решателя и хэша модели. Региональное разбиение должно быть полным.
Совпадение базового импеданса само по себе не валидирует карту чувствительности.


In [ ]:
import hashlib
import json
import os
from pathlib import Path

import numpy as np

from ttrkg_analysis import validate_fractional_operator

test = validate_fractional_operator(["soft_tissue_wall", "lung", "other"], [0.65, 0.45, -0.10], complete_partition=True)
assert test["has_negative"] and abs(test["sum"] - 1.0) < 1e-12
REAL_MODE = os.environ.get("KALMYKOV_RUN_REAL", "0") == "1"
print("40.04 synthetic_self_test: passed")


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def key(item):
    return tuple(item.get(field) for field in ['experiment_id', 'subject_id', 'record_id', 'configuration_id', 'montage_id', 'side_montage_id', 'side_size_mm', 'channel_state', 'mode'])


def require_sha256(value, label):
    if not isinstance(value, str) or len(value) != 64 or any(char not in "0123456789abcdef" for char in value.lower()):
        raise RuntimeError(f"Некорректный SHA-256: {label}")


if not REAL_MODE:
    print("40.04 real_data_status: blocked_until_exact_complete_fem_operator_index")
else:
    config_path = Path(os.environ["KALMYKOV_EXP02_CONFIG"]).expanduser().resolve()
    config = json.loads(config_path.read_text(encoding="utf-8"))
    fem_spec = config.get("fem_analysis", {})
    if fem_spec.get("status") != "accepted" or not fem_spec.get("operator_index_manifest"):
        raise RuntimeError("Не принят индекс FEM-операторов")
    derived_root = Path(config["derived_root"]).expanduser().resolve()
    analysis_dir = derived_root / "exp02" / "analysis"
    measured_path = analysis_dir / "40.01_ttrkg_ensembles.json"
    measured = json.loads(measured_path.read_text(encoding="utf-8"))
    if measured.get("status") != "accepted_input_conditional_ttrkg_ensembles" or not measured.get("ensembles"):
        raise RuntimeError("Нет принятого непустого артефакта ТТРКГ")
    measured_by_key = {key(item): item for item in measured["ensembles"]}
    if len(measured_by_key) != len(measured["ensembles"]):
        raise RuntimeError("Повторный ключ в измеренных ансамблях")

    index_path = Path(fem_spec["operator_index_manifest"]).expanduser().resolve()
    index = json.loads(index_path.read_text(encoding="utf-8"))
    if index.get("status") != "accepted" or index.get("experiment_id") != "exp02":
        raise RuntimeError("Индекс FEM не принят или относится к другому эксперименту")
    entries = index.get("entries", [])
    index_by_key = {key(item): item for item in entries}
    if not entries or len(index_by_key) != len(entries) or set(index_by_key) != set(measured_by_key):
        raise RuntimeError("Индекс FEM должен один-к-одному совпадать с измеренными ансамблями")
    checked = []
    for item_key, measured_item in measured_by_key.items():
        entry = index_by_key[item_key]
        if entry.get("channel_state") not in ['ttrkg_channel_1_with_side_channel_2_connected']:
            raise RuntimeError("Недопустимое состояние каналов в FEM-индексе")
        operator_path = Path(entry["operator_manifest"]).expanduser().resolve()
        operator = json.loads(operator_path.read_text(encoding="utf-8"))
        if operator.get("status") != "accepted" or key(operator) != item_key:
            raise RuntimeError("FEM-оператор не принят или его ключ не совпал с записью")
        if float(operator.get("frequency_hz", 0.0)) <= 0 or operator.get("respiratory_state") != measured_item["mode"]:
            raise RuntimeError("Не совпали частота или дыхательное состояние FEM")
        geometry = operator.get("geometry", {})
        if geometry.get("segmentation_method") != "manual_Inobitec":
            raise RuntimeError("Канонической считается ручная сегментация Inobitec")
        require_sha256(geometry.get("geometry_sha256"), "geometry_sha256")
        require_sha256(operator.get("electrode_coordinates_sha256"), "electrode_coordinates_sha256")
        require_sha256(operator.get("material_properties_sha256"), "material_properties_sha256")
        solver = operator.get("solver", {})
        if not solver.get("name") or not solver.get("version"):
            raise RuntimeError("Не указаны решатель и его версия")
        require_sha256(solver.get("model_sha256"), "solver.model_sha256")
        if operator.get("mesh_convergence", {}).get("status") != "accepted":
            raise RuntimeError("Не принята сеточная сходимость")
        if operator.get("base_impedance_check", {}).get("status") != "accepted":
            raise RuntimeError("Не принята проверка базового импеданса")
        if operator.get("complete_regional_partition") is not True:
            raise RuntimeError("Неполный региональный FEM-оператор не допускается")
        names = operator.get("region_names", [])
        values = operator.get("fractional_sensitivities", [])
        diagnostics = validate_fractional_operator(names, values, complete_partition=True, sum_tolerance=float(operator.get("homogeneity_sum_tolerance", 1e-6)))
        if "exp02" == "exp03" and operator["channel_state"] == "ttrkg_channels_1_and_2_connected":
            coupling = operator.get("device_coupling_model", {})
            if coupling.get("status") != "accepted" or coupling.get("validation_analysis") != "40.12":
                raise RuntimeError("Двухканальная FEM-модель РНЦХ не связана с принятым результатом 40.12")
            require_sha256(coupling.get("validation_artifact_sha256"), "device_coupling_model.validation_artifact_sha256")
        checked.append({
            **{field: measured_item.get(field) for field in ['experiment_id', 'subject_id', 'record_id', 'configuration_id', 'montage_id', 'side_montage_id', 'side_size_mm', 'channel_state', 'mode']},
            "frequency_hz": float(operator["frequency_hz"]), "region_names": names,
            "fractional_sensitivities": values, "sensitivity_sum": diagnostics["sum"],
            "has_negative_sensitivity": diagnostics["has_negative"],
            "operator_manifest_sha256": sha256_file(operator_path),
            "geometry_sha256": geometry["geometry_sha256"],
            "electrode_coordinates_sha256": operator["electrode_coordinates_sha256"],
            "material_properties_sha256": operator["material_properties_sha256"],
            "solver": solver,
        })
    artifact = {
        "schema_version": 2, "analysis": "40.04_checked_fem_operator",
        "status": "accepted_complete_fem_operator_contract",
        "measured_artifact_sha256": sha256_file(measured_path),
        "operator_index_sha256": sha256_file(index_path), "operators": checked,
        "limitations": ["base_impedance_agreement_does_not_validate_regional_sensitivity", "operator_is_configuration_specific"],
    }
    out_path = analysis_dir / "40.04_fem_operator_checked.json"
    out_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("40.04 real_data_status: artifact_written", out_path)


## Граница результата

Проверяется полнота и прослеживаемость внешнего оператора, а не истинность всех
его региональных производных. Отрицательная тетраполярная чувствительность
допустима; запрет `0 ≤ Sᵢ ≤ 1` не вводится.
